# 02 - Multi-level Richardson extrapolation: full pipeline (OFFLINE)

Runs end to end with **no network access**. Attach the output of `01_prefetch_online`
as an input dataset first.

Produces, under `/kaggle/working/outputs`:
* `results/*.csv` - every number, one table per study
* `figures/*.pdf` - one caption-less vector figure per result
* `figures/data/*.csv` - the exact numbers behind each figure, for restyling later

**The GPU stage is resumable.** Re-running skips configurations already in
`results/fid_results.csv`, so if the session dies, just run it again.


In [ ]:
# ---- the one thing you may need to change -------------------------------
PREFETCH_NAME = None   # e.g. '01-prefetch-online'; None = auto-discover

# ---- experiment budget --------------------------------------------------
N_IMAGES  = 10_000               # per FID (paper uses 50k; see note below)
NFE_GRID  = (6, 8, 10, 12, 16, 20)
BATCH     = 256                  # raise on a 24 GB card
QUICK_CPU = False
RUN_GPU   = True
INCLUDE_REUSE   = True           # reuse-vs-exact arm
DO_RHO_SEARCH   = True

OUTDIR = '/kaggle/working/outputs'


## Locate the prefetched input

Searches the documented path first, then falls back to scanning `/kaggle/input`
for the `manifest.json` sentinel, so a differing mount layout is not fatal.

In [ ]:
import os, sys, glob, json, time, subprocess

def find_prefetch():
    cands = []
    if PREFETCH_NAME:
        cands += [f'/kaggle/input/notebooks/raihanzahin/{PREFETCH_NAME}',
                  f'/kaggle/input/{PREFETCH_NAME}']
    cands += sorted(glob.glob('/kaggle/input/notebooks/*/*'))
    cands += sorted(glob.glob('/kaggle/input/*'))
    cands += ['/kaggle/working']
    for c in cands:
        if os.path.exists(os.path.join(c, 'manifest.json')):
            return c
    raise SystemExit(
        'Could not find the prefetch output (no manifest.json).\n'
        'Add notebook 01 output via + Add Input, then set PREFETCH_NAME.\n'
        'Searched:\n  ' + '\n  '.join(cands[:25]))

IN = find_prefetch()
man = json.load(open(f'{IN}/manifest.json'))
print('prefetch:', IN)
print('created :', man['created'])
for k, v in man['assets'].items():
    print(f"  {k:<40} {v['bytes']/2**20:8.1f} MB")

REPO = f'{IN}/repo/project'
EDM  = f'{IN}/third_party/edm'
ASSETS = f'{IN}/assets'
for p in (REPO, EDM, ASSETS):
    assert os.path.isdir(p), p


In [ ]:
# Offline wheel install (no-op if the image already has everything).
w = f'{IN}/wheels'
if os.path.isdir(w) and os.listdir(w):
    subprocess.run(f'pip install -q --no-index --find-links {w} '
                   f'click tqdm requests psutil', shell=True)

sys.path.insert(0, f'{REPO}/src')
sys.path.insert(0, EDM)
os.environ['EDM_ROOT'] = EDM

import torch, numpy as np, matplotlib
matplotlib.use('Agg')
import mlrx
print('mlrx', mlrx.__version__)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| devices', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  {p.total_memory/2**30:.1f} GB')


## Correctness gate

The test suite includes a bit-exact comparison against the authors' published
sampler. If this fails, nothing downstream is worth reading.

In [ ]:
r = subprocess.run(f'python -m pytest {REPO}/tests -q',
                   shell=True, text=True, capture_output=True,
                   env={**os.environ, 'PYTHONPATH': f'{REPO}/src'})
print(r.stdout[-3000:] or r.stderr[-3000:])
assert r.returncode == 0, 'tests failed - stop here'


## Stage 1 - CPU experiments

No GPU, no assets. About a minute. This is where most of the project's findings
come from, because the analytic problem has a known exact solution and error can
be *measured* rather than proxied by FID.

In [ ]:
from mlrx.pipeline import run_cpu_stage
written, summary = run_cpu_stage(OUTDIR, quick=QUICK_CPU)
print(json.dumps(summary, indent=2))


In [ ]:
import pandas as pd
ab = pd.read_csv(f'{OUTDIR}/results/reuse_ablation_order.csv')
print('HEADLINE - fitted local order by level count and reuse mode\n')
for prob, g in ab.groupby('problem'):
    print(prob)
    print(g.pivot_table(index='n_levels', columns='reuse_mode',
                        values='fitted_local_order')
           .round(2).to_string(), '\n')


## Stage 2 - throughput probe

One small run, timed, to project the real cost of the sweep on *this* hardware
before committing hours to it. If the projection exceeds the session limit,
lower `N_IMAGES` or trim `NFE_GRID` and re-run this cell.

In [ ]:
from mlrx.runner import GPUContext, Config, run_config
from mlrx.experiments import sweeps

CTX = dict(network_pkl=f'{ASSETS}/edm-cifar10-32x32-cond-vp.pkl',
           detector_pkl=f'{ASSETS}/inception-2015-12-05.pkl',
           ref_npz=f'{ASSETS}/cifar10-32x32.npz',
           edm_root=EDM, batch_size=BATCH)

if RUN_GPU and torch.cuda.is_available():
    ctx = GPUContext(device='cuda:0', **CTX)
    probe = Config(method='euler', num_steps=10, n_images=1024, tag='probe')
    t0 = time.time(); m = run_config(probe, ctx); dt = time.time() - t0
    rate = probe.n_images * probe.expected_nfe / dt
    print(f'probe: FID {m["fid"]:.2f} on {probe.n_images} imgs in {dt:.0f}s')
    print(f'measured throughput: {rate:,.0f} image-NFE/s')

    cfgs = sweeps.build_all(n_images=N_IMAGES, nfe_grid=NFE_GRID,
                            include_reuse=INCLUDE_REUSE)
    est = sweeps.estimate_cost(cfgs, throughput_img_nfe_per_s=rate,
                               n_gpus=max(1, torch.cuda.device_count()))
    print(f"\n{est['n_configs']} configs, {est['image_nfe']/1e6:.1f}M image-NFE")
    print(f"projected: {est['gpu_hours']:.1f} GPU-h, "
          f"{est['wall_hours']:.1f} h wall")
    if est['wall_hours'] > 10.5:
        print('\n*** exceeds a comfortable session budget. Either lower N_IMAGES,\n'
              '    trim NFE_GRID, or just run this notebook twice - the sweep\n'
              '    resumes from results/fid_results.csv. ***')


## Stage 3 - the FID sweep

Resumable. Every completed configuration is appended to `results/fid_results.csv`
immediately and skipped on a later run.

In [ ]:
if RUN_GPU and torch.cuda.is_available():
    from mlrx.pipeline import run_gpu_stage
    figs, df = run_gpu_stage(
        OUTDIR, CTX, n_images=N_IMAGES, nfe_grid=NFE_GRID,
        n_gpus=max(1, torch.cuda.device_count()),
        include_reuse=INCLUDE_REUSE, do_rho_search=DO_RHO_SEARCH)
    print(f'\n{len(df)} configurations recorded')
else:
    print('GPU stage skipped')


## Results

In [ ]:
import pandas as pd
fp = f'{OUTDIR}/results/fid_results.csv'
if os.path.exists(fp):
    df = pd.read_csv(fp)
    gate = df[(df.nfe == 10) & (df.tag.isin(['panel','validity','seedblock']))]
    if len(gate):
        print('Validation gate, NFE = 10:\n')
        print(gate.groupby('method')['fid'].agg(['mean','std','count'])
                  .round(3).to_string())
        print('\nPaper (50k images): Euler 15.88 | Heun 14.46 | RX-Euler 4.35 | RX+EDM 4.26')
        print('Ours use 10k images, so they read high - FID is biased upward at\n'
              'smaller sample counts. Compare our methods against each other,\n'
              'not against the published absolute values.')


In [ ]:
print('figures:')
for f in sorted(glob.glob(f'{OUTDIR}/figures/*.pdf')):
    print(f'  {os.path.basename(f):<48} {os.path.getsize(f)/1024:6.1f} KB')
print('\ntables:')
for f in sorted(glob.glob(f'{OUTDIR}/results/*.csv')):
    print(f'  {os.path.basename(f):<48} {sum(1 for _ in open(f))-1:6d} rows')


## Package the outputs

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/mlrx_outputs', 'zip', OUTDIR)
print('wrote /kaggle/working/mlrx_outputs.zip '
      f'({os.path.getsize("/kaggle/working/mlrx_outputs.zip")/2**20:.1f} MB)')
